In [ ]:
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier


# ============================================================
# 1. LOCATE AND LOAD THE DATA
# ============================================================

# Works in either:
# 1. A Kaggle notebook with the Titanic dataset attached
# 2. A local project with files inside a folder named "data"

kaggle_data_path = Path("/kaggle/input/titanic")
local_data_path = Path("data")

if kaggle_data_path.exists():
    data_path = kaggle_data_path
    output_path = Path("/kaggle/working")
else:
    data_path = local_data_path
    output_path = Path(".")

train_path = data_path / "train.csv"
test_path = data_path / "test.csv"

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError(
        f"Could not find train.csv and test.csv in: {data_path.resolve()}\n"
        "Place the files inside a folder named 'data', or attach the "
        "Titanic dataset to your Kaggle notebook."
    )

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("Training data shape:", train.shape)
print("Test data shape:", test.shape)
print("\nTraining columns:")
print(train.columns.tolist())

In [ ]:
# ============================================================
# 2. QUICK DATA CHECKS
# ============================================================

print("\nTarget distribution:")
print(train["Survived"].value_counts(normalize=True).round(3))

print("\nMissing values in training data:")
print(
    train.isna()
    .sum()
    .sort_values(ascending=False)
    .loc[lambda x: x > 0]
)

print("\nMissing values in test data:")
print(
    test.isna()
    .sum()
    .sort_values(ascending=False)
    .loc[lambda x: x > 0]
)


# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a small number of interpretable Titanic features.

    This function does not use Survived, so it can safely be applied
    to both the training data and the Kaggle test data.
    """
    df = df.copy()

    # Extract titles such as Mr, Mrs, Miss, and Master from Name.
    df["Title"] = (
        df["Name"]
        .str.extract(r",\s*([^.]*)\.", expand=False)
        .str.strip()
    )

    # Standardize equivalent titles.
    df["Title"] = df["Title"].replace(
        {
            "Mlle": "Miss",
            "Ms": "Miss",
            "Mme": "Mrs",
        }
    )

    # Combine uncommon titles to avoid creating many tiny categories.
    common_titles = ["Mr", "Mrs", "Miss", "Master"]
    df["Title"] = df["Title"].where(
        df["Title"].isin(common_titles),
        "Rare"
    )

    # Family members aboard, including the passenger.
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

    # Whether the passenger was traveling alone.
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    # Use the first letter of Cabin as an approximate deck.
    df["Deck"] = df["Cabin"].fillna("Unknown").str[0]

    # Approximate fare per family member.
    df["FarePerPerson"] = df["Fare"] / df["FamilySize"]

    return df


train_features = add_features(train)
test_features = add_features(test)


# ============================================================
# 4. DEFINE THE MODELING DATA
# ============================================================

feature_columns = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "Title",
    "FamilySize",
    "IsAlone",
    "Deck",
    "FarePerPerson",
]

numeric_features = [
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize",
    "IsAlone",
    "FarePerPerson",
]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked",
    "Title",
    "Deck",
]

X = train_features[feature_columns]
y = train_features["Survived"]

X_test = test_features[feature_columns]


# ============================================================
# 5. PREPROCESSING
# ============================================================

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)


##### EXP-001 - XGB

In [2]:

# ============================================================
# 6. XGBOOST MODEL
# ============================================================

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=2,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.10,
    reg_lambda=2.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", model),
    ]
)


# ============================================================
# 7. CROSS-VALIDATION
# ============================================================

cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

cv_scores = cross_val_score(
    pipeline,
    X,
    y,
    cv=cross_validation,
    scoring="accuracy",
)

print("\nCross-validation scores:")
print(cv_scores.round(4))

print(f"\nMean CV accuracy: {cv_scores.mean():.4f}")
print(f"CV standard deviation: {cv_scores.std():.4f}")


# ============================================================
# 8. TRAIN ON ALL LABELED DATA
# ============================================================

pipeline.fit(X, y)


# ============================================================
# 9. PREDICT THE KAGGLE TEST DATA
# ============================================================

test_predictions = pipeline.predict(X_test).astype(int)

print("\nPrediction distribution:")
print(pd.Series(test_predictions).value_counts().sort_index())


# ============================================================
# 10. CREATE THE SUBMISSION
# ============================================================

submission = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "Survived": test_predictions,
    }
)

submission_path = output_path / "submission_xgb_baseline.csv"
submission.to_csv(submission_path, index=False)

print("\nSubmission preview:")
print(submission.head(10))

print("\nSubmission shape:", submission.shape)
print("Submission saved to:", submission_path.resolve())


# Final safety checks
assert submission.columns.tolist() == ["PassengerId", "Survived"]
assert len(submission) == len(test)
assert submission["PassengerId"].equals(test["PassengerId"])
assert submission["Survived"].isin([0, 1]).all()

print("\nAll submission checks passed.")

Training data shape: (891, 12)
Test data shape: (418, 11)

Training columns:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Target distribution:
Survived
0    0.616
1    0.384
Name: proportion, dtype: float64

Missing values in training data:
Cabin       687
Age         177
Embarked      2
dtype: int64

Missing values in test data:
Cabin    327
Age       86
Fare       1
dtype: int64

Cross-validation scores:
[0.8659 0.8483 0.8034 0.8258 0.8427]

Mean CV accuracy: 0.8372
CV standard deviation: 0.0212

Prediction distribution:
0    262
1    156
Name: count, dtype: int64

Submission preview:
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
5          897         0
6          898         1
7          899         0
8          900         1
9          901         0

Submission shape: (418, 2)
Submission saved to: C:\Users\

##### EXP-002 - Logistic

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

logistic_model = LogisticRegression(
    C=1.0,
    max_iter=2000,
    random_state=42,
)

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", logistic_model),
    ]
)

logistic_cv_scores = cross_val_score(
    logistic_pipeline,
    X,
    y,
    cv=cross_validation,
    scoring="accuracy",
)

print("Logistic regression CV scores:")
print(logistic_cv_scores.round(4))

print(f"\nMean CV accuracy: {logistic_cv_scores.mean():.4f}")
print(f"CV standard deviation: {logistic_cv_scores.std():.4f}")

logistic_pipeline.fit(X, y)

logistic_predictions = logistic_pipeline.predict(X_test).astype(int)

logistic_submission = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "Survived": logistic_predictions,
    }
)

logistic_submission_path = output_path / "submission_logistic.csv"
logistic_submission.to_csv(logistic_submission_path, index=False)

print("\nPrediction distribution:")
print(
    logistic_submission["Survived"]
    .value_counts()
    .sort_index()
)

print("\nSubmission preview:")
print(logistic_submission.head())

print("\nSaved to:")
print(logistic_submission_path.resolve())

assert logistic_submission.columns.tolist() == [
    "PassengerId",
    "Survived",
]
assert len(logistic_submission) == len(test)
assert logistic_submission["Survived"].isin([0, 1]).all()

Logistic regression CV scores:
[0.8436 0.8258 0.7978 0.8315 0.8427]

Mean CV accuracy: 0.8283
CV standard deviation: 0.0167

Prediction distribution:
Survived
0    250
1    168
Name: count, dtype: int64

Submission preview:
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1

Saved to:
C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\01-Titanic - Machine Learning from Disaster\submission_logistic.csv


##### EXP-003 - RF

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
import pandas as pd


# ============================================================
# EXP-003: RANDOM FOREST
# ============================================================

random_forest_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=5,
    min_samples_split=8,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)

random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", random_forest_model),
    ]
)

rf_cv_scores = cross_val_score(
    random_forest_pipeline,
    X,
    y,
    cv=cross_validation,
    scoring="accuracy",
)

print("Random Forest CV scores:")
print(rf_cv_scores.round(4))

print(f"\nMean CV accuracy: {rf_cv_scores.mean():.4f}")
print(f"CV standard deviation: {rf_cv_scores.std():.4f}")


# Train using all labeled passengers
random_forest_pipeline.fit(X, y)


# Predict the Kaggle test set
rf_predictions = (
    random_forest_pipeline
    .predict(X_test)
    .astype(int)
)


# Create submission
rf_submission = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "Survived": rf_predictions,
    }
)

rf_submission_path = (
    output_path / "submission_random_forest.csv"
)

rf_submission.to_csv(
    rf_submission_path,
    index=False,
)


# Output checks
print("\nPrediction counts:")
print(
    rf_submission["Survived"]
    .value_counts()
    .sort_index()
)

print("\nSubmission preview:")
display(rf_submission.head())

print("\nSaved to:")
print(rf_submission_path.resolve())


assert rf_submission.columns.tolist() == [
    "PassengerId",
    "Survived",
]

assert len(rf_submission) == len(test)

assert rf_submission["PassengerId"].equals(
    test["PassengerId"]
)

assert rf_submission["Survived"].isin([0, 1]).all()

Random Forest CV scores:
[0.838  0.8202 0.8202 0.8315 0.8483]

Mean CV accuracy: 0.8316
CV standard deviation: 0.0108

Prediction counts:
Survived
0    259
1    159
Name: count, dtype: int64

Submission preview:


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1



Saved to:
C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\01-Titanic - Machine Learning from Disaster\submission_random_forest.csv


In [10]:
model_comparison = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "logistic": logistic_predictions,
        "random_forest": rf_predictions,
    }
)

model_comparison["disagree"] = (
    model_comparison["logistic"]
    != model_comparison["random_forest"]
)

print(
    "Differing test predictions:",
    model_comparison["disagree"].sum(),
)

display(
    model_comparison[
        model_comparison["disagree"]
    ].head(20)
)

Differing test predictions: 15


,PassengerId,logistic,random_forest,disagree
1,893,1,0,True
18,910,1,0,True
32,924,0,1,True
41,933,1,0,True
73,965,1,0,True
75,967,1,0,True
127,1019,0,1,True
146,1038,1,0,True
181,1073,1,0,True
242,1134,1,0,True


In [13]:
test_disagreements = test_features.loc[
    rf_predictions != logistic_predictions,
    [
        "PassengerId",
        "Name",
        "Sex",
        "Age",
        "Pclass",
        "Fare",
        "SibSp",
        "Parch",
        "Embarked",
        "Title",
        "FamilySize",
        "IsAlone",
        "Deck",
    ],
].copy()

test_disagreements["logistic_prediction"] = logistic_predictions[
    rf_predictions != logistic_predictions
]

test_disagreements["random_forest_prediction"] = rf_predictions[
    rf_predictions != logistic_predictions
]

test_disagreements["logistic_probability"] = (
    logistic_pipeline.predict_proba(X_test)[:, 1][
        rf_predictions != logistic_predictions
    ]
)

test_disagreements["random_forest_probability"] = (
    random_forest_pipeline.predict_proba(X_test)[:, 1][
        rf_predictions != logistic_predictions
    ]
)

display(
    test_disagreements.sort_values(
        "random_forest_probability",
        ascending=False,
    )
)

,PassengerId,Name,Sex,Age,Pclass,Fare,SibSp,Parch,Embarked,Title,FamilySize,IsAlone,Deck,logistic_prediction,random_forest_prediction,logistic_probability,random_forest_probability
127,1019,"McCoy, Miss. Alicia",female,NaN,3,23.2500,2,0,Q,Miss,3,0,U,0,1,0.489544,0.651755
32,924,"Dean, Mrs. Bertram (Eva Georgetta Light)",female,33.0,3,20.5750,1,2,S,Mrs,4,0,U,0,1,0.474766,0.549814
293,1185,"Dodge, Dr. Washington",male,53.0,1,81.8583,1,1,S,Rare,3,0,A,0,1,0.294045,0.516752
1,893,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,3,7.0000,1,0,S,Mrs,2,0,U,1,0,0.560250,0.499876
18,910,"Ilmakangas, Miss. Ida Livija",female,27.0,3,7.9250,1,0,S,Miss,2,0,U,1,0,0.541840,0.487551
181,1073,"Compton, Mr. Alexander Taylor Jr",male,37.0,1,83.1583,1,1,C,Mr,3,0,E,1,0,0.558077,0.462081
339,1231,"Betros, Master. Seman",male,NaN,3,7.2292,0,0,C,Master,1,1,U,1,0,0.712237,0.423276
73,965,"Ovies y Rodriguez, Mr. Servando",male,28.5,1,27.7208,0,0,C,Mr,1,1,D,1,0,0.638612,0.416571
41,933,"Franklin, Mr. Thomas Parham",male,NaN,1,26.5500,0,0,S,Mr,1,1,D,1,0,0.540525,0.409378
242,1134,"Spedden, Mr. Frederic Oakley",male,45.0,1,134.5000,1,1,C,Mr,3,0,E,1,0,0.555280,0.407859


In [14]:
share_columns = [
    "PassengerId",
    "Sex",
    "Age",
    "Pclass",
    "Fare",
    "Title",
    "FamilySize",
    "IsAlone",
    "Deck",
    "logistic_prediction",
    "random_forest_prediction",
    "logistic_probability",
    "random_forest_probability",
]

print(
    test_disagreements[share_columns]
    .round(
        {
            "Age": 1,
            "Fare": 2,
            "logistic_probability": 3,
            "random_forest_probability": 3,
        }
    )
    .to_string(index=False)
)

 PassengerId    Sex  Age  Pclass   Fare  Title  FamilySize  IsAlone Deck  logistic_prediction  random_forest_prediction  logistic_probability  random_forest_probability
         893 female 47.0       3   7.00    Mrs           2        0    U                    1                         0                 0.560                      0.500
         910 female 27.0       3   7.92   Miss           2        0    U                    1                         0                 0.542                      0.488
         924 female 33.0       3  20.58    Mrs           4        0    U                    0                         1                 0.475                      0.550
         933   male  NaN       1  26.55     Mr           1        1    D                    1                         0                 0.541                      0.409
         965   male 28.5       1  27.72     Mr           1        1    D                    1                         0                 0.639              

##### EXP-004

In [16]:
# ============================================================
# EXP-004: LOGISTIC REGRESSION WITH INTERACTION FEATURES
# ============================================================

import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline


# ------------------------------------------------------------
# 1. ADD TARGETED INTERACTION FEATURES
# ------------------------------------------------------------

def add_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Passenger class may affect men and women differently.
    df["Sex_Pclass"] = (
        df["Sex"].astype(str)
        + "_class_"
        + df["Pclass"].astype(str)
    )

    # Titles such as Mr, Mrs, Miss, and Master interact strongly with sex.
    df["Sex_Title"] = (
        df["Sex"].astype(str)
        + "_"
        + df["Title"].astype(str)
    )

    # Class may affect different titles differently.
    df["Title_Pclass"] = (
        df["Title"].astype(str)
        + "_class_"
        + df["Pclass"].astype(str)
    )

    return df


train_interactions = add_interaction_features(train_features)
test_interactions = add_interaction_features(test_features)


# ------------------------------------------------------------
# 2. UPDATE FEATURE LISTS
# ------------------------------------------------------------

interaction_features = [
    "Sex_Pclass",
    "Sex_Title",
    "Title_Pclass",
]

interaction_feature_columns = (
    feature_columns
    + interaction_features
)

interaction_categorical_features = (
    categorical_features
    + interaction_features
)

X_interactions = train_interactions[
    interaction_feature_columns
]

X_test_interactions = test_interactions[
    interaction_feature_columns
]


# ------------------------------------------------------------
# 3. CREATE A NEW PREPROCESSOR
# ------------------------------------------------------------

interaction_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            interaction_categorical_features,
        ),
    ]
)


# ------------------------------------------------------------
# 4. LOGISTIC REGRESSION
# ------------------------------------------------------------

logistic_interaction_model = LogisticRegression(
    C=1.0,
    max_iter=2000,
    random_state=42,
)

logistic_interaction_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            interaction_preprocessor,
        ),
        (
            "model",
            logistic_interaction_model,
        ),
    ]
)


# ------------------------------------------------------------
# 5. CROSS-VALIDATION
# ------------------------------------------------------------

logistic_interaction_cv_scores = cross_val_score(
    logistic_interaction_pipeline,
    X_interactions,
    y,
    cv=cross_validation,
    scoring="accuracy",
)

print("Interaction Logistic CV scores:")
print(
    logistic_interaction_cv_scores.round(4)
)

print(
    f"\nMean CV accuracy: "
    f"{logistic_interaction_cv_scores.mean():.4f}"
)

print(
    f"CV standard deviation: "
    f"{logistic_interaction_cv_scores.std():.4f}"
)


# ------------------------------------------------------------
# 6. TRAIN AND PREDICT
# ------------------------------------------------------------

logistic_interaction_pipeline.fit(
    X_interactions,
    y,
)

logistic_interaction_predictions = (
    logistic_interaction_pipeline
    .predict(X_test_interactions)
    .astype(int)
)


# ------------------------------------------------------------
# 7. CREATE SUBMISSION
# ------------------------------------------------------------

interaction_submission = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "Survived": logistic_interaction_predictions,
    }
)

interaction_submission_path = (
    output_path
    / "submission_logistic_interactions.csv"
)

interaction_submission.to_csv(
    interaction_submission_path,
    index=False,
)

print("\nPrediction counts:")
print(
    interaction_submission["Survived"]
    .value_counts()
    .sort_index()
)

print("\nSubmission preview:")
display(interaction_submission.head())

print("\nSaved to:")
print(interaction_submission_path.resolve())


# ------------------------------------------------------------
# 8. SAFETY CHECKS
# ------------------------------------------------------------

assert interaction_submission.columns.tolist() == [
    "PassengerId",
    "Survived",
]

assert len(interaction_submission) == len(test)

assert interaction_submission[
    "PassengerId"
].equals(test["PassengerId"])

assert interaction_submission[
    "Survived"
].isin([0, 1]).all()

Interaction Logistic CV scores:
[0.8268 0.8539 0.7978 0.8427 0.8483]

Mean CV accuracy: 0.8339
CV standard deviation: 0.0202

Prediction counts:
Survived
0    261
1    157
Name: count, dtype: int64

Submission preview:


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1



Saved to:
C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\01-Titanic - Machine Learning from Disaster\submission_logistic_interactions.csv


In [17]:
comparison_exp_004 = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "logistic_baseline": logistic_predictions,
        "logistic_interactions": logistic_interaction_predictions,
        "random_forest": rf_predictions,
    }
)

print(
    "Interaction logistic vs baseline logistic:",
    (
        comparison_exp_004["logistic_interactions"]
        != comparison_exp_004["logistic_baseline"]
    ).sum(),
)

print(
    "Interaction logistic vs random forest:",
    (
        comparison_exp_004["logistic_interactions"]
        != comparison_exp_004["random_forest"]
    ).sum(),
)

Interaction logistic vs baseline logistic: 11
Interaction logistic vs random forest: 16


##### EXP-005

In [20]:
# ============================================================
# EXP-005: RANDOM FOREST WITH GROUP-BASED AGE IMPUTATION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline


# ------------------------------------------------------------
# 1. CUSTOM AGE IMPUTER
# ------------------------------------------------------------

class GroupedAgeImputer(BaseEstimator, TransformerMixin):
    """
    Impute missing Age using medians learned from training data.

    Fallback order:
    1. Title + Pclass median
    2. Title median
    3. Pclass median
    4. Overall median
    """

    def fit(self, X, y=None):
        X = X.copy()

        self.title_class_medians_ = (
            X.groupby(["Title", "Pclass"])["Age"]
            .median()
            .to_dict()
        )

        self.title_medians_ = (
            X.groupby("Title")["Age"]
            .median()
            .to_dict()
        )

        self.class_medians_ = (
            X.groupby("Pclass")["Age"]
            .median()
            .to_dict()
        )

        self.overall_median_ = X["Age"].median()

        return self

    def transform(self, X):
        X = X.copy()

        missing_age_rows = X["Age"].isna()

        for row_index in X.index[missing_age_rows]:
            title = X.at[row_index, "Title"]
            passenger_class = X.at[row_index, "Pclass"]

            estimated_age = self.title_class_medians_.get(
                (title, passenger_class),
                np.nan,
            )

            if pd.isna(estimated_age):
                estimated_age = self.title_medians_.get(
                    title,
                    np.nan,
                )

            if pd.isna(estimated_age):
                estimated_age = self.class_medians_.get(
                    passenger_class,
                    np.nan,
                )

            if pd.isna(estimated_age):
                estimated_age = self.overall_median_

            X.at[row_index, "Age"] = estimated_age

        return X


# ------------------------------------------------------------
# 2. SAME RANDOM FOREST SETTINGS AS EXP-003
# ------------------------------------------------------------

rf_grouped_age_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=5,
    min_samples_split=8,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)


# ------------------------------------------------------------
# 3. PIPELINE
# ------------------------------------------------------------

rf_grouped_age_pipeline = Pipeline(
    steps=[
        (
            "grouped_age_imputer",
            GroupedAgeImputer(),
        ),
        (
            "preprocessing",
            preprocessor,
        ),
        (
            "model",
            rf_grouped_age_model,
        ),
    ]
)


# ------------------------------------------------------------
# 4. CROSS-VALIDATION
# ------------------------------------------------------------

rf_grouped_age_cv_scores = cross_val_score(
    rf_grouped_age_pipeline,
    X,
    y,
    cv=cross_validation,
    scoring="accuracy",
)

print("Grouped-Age Random Forest CV scores:")
print(rf_grouped_age_cv_scores.round(4))

print(
    f"\nMean CV accuracy: "
    f"{rf_grouped_age_cv_scores.mean():.4f}"
)

print(
    f"CV standard deviation: "
    f"{rf_grouped_age_cv_scores.std():.4f}"
)


# ------------------------------------------------------------
# 5. TRAIN ON ALL LABELED DATA
# ------------------------------------------------------------

rf_grouped_age_pipeline.fit(X, y)


# ------------------------------------------------------------
# 6. PREDICT TEST DATA
# ------------------------------------------------------------

rf_grouped_age_predictions = (
    rf_grouped_age_pipeline
    .predict(X_test)
    .astype(int)
)


# ------------------------------------------------------------
# 7. CREATE SUBMISSION
# ------------------------------------------------------------

rf_grouped_age_submission = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "Survived": rf_grouped_age_predictions,
    }
)

rf_grouped_age_submission_path = (
    output_path
    / "submission_rf_grouped_age.csv"
)

rf_grouped_age_submission.to_csv(
    rf_grouped_age_submission_path,
    index=False,
)

print("\nPrediction counts:")
print(
    rf_grouped_age_submission["Survived"]
    .value_counts()
    .sort_index()
)

print("\nSubmission preview:")
display(rf_grouped_age_submission.head())

print("\nSaved to:")
print(rf_grouped_age_submission_path.resolve())


# ------------------------------------------------------------
# 8. SAFETY CHECKS
# ------------------------------------------------------------

assert rf_grouped_age_submission.columns.tolist() == [
    "PassengerId",
    "Survived",
]

assert len(rf_grouped_age_submission) == len(test)

assert rf_grouped_age_submission[
    "PassengerId"
].equals(test["PassengerId"])

assert rf_grouped_age_submission[
    "Survived"
].isin([0, 1]).all()

Grouped-Age Random Forest CV scores:
[0.838  0.8146 0.8258 0.8315 0.8371]

Mean CV accuracy: 0.8294
CV standard deviation: 0.0086

Prediction counts:
Survived
0    255
1    163
Name: count, dtype: int64

Submission preview:


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1



Saved to:
C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\01-Titanic - Machine Learning from Disaster\submission_rf_grouped_age.csv


In [21]:
exp_005_comparison = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"],
        "rf_baseline": rf_predictions,
        "rf_grouped_age": rf_grouped_age_predictions,
    }
)

exp_005_comparison["disagree"] = (
    exp_005_comparison["rf_baseline"]
    != exp_005_comparison["rf_grouped_age"]
)

print(
    "Differing predictions versus EXP-003:",
    exp_005_comparison["disagree"].sum(),
)

display(
    exp_005_comparison[
        exp_005_comparison["disagree"]
    ]
)

Differing predictions versus EXP-003: 4


,PassengerId,rf_baseline,rf_grouped_age,disagree
1,893,0,1,True
244,1136,0,1,True
339,1231,0,1,True
344,1236,0,1,True


##### EXP-006

The ordinary StratifiedKFold validation may be optimistic because related passengers can appear in both the training and validation portions of a fold.

For example, the model could train on one member of the Andersson family and validate on another member with similar:

Surname
Fare
Ticket characteristics
Family size
Passenger class

Group-aware validation forces likely family members to remain together.

What remains constant
The EXP-003 Random Forest
All model parameters
All features
All preprocessing
Five validation folds
Accuracy as the evaluation metric
The complete Titanic training dataset
What changes

Only the validation split:

Current method: StratifiedKFold
New method: StratifiedGroupKFold
Group definition: surname plus family size
Passengers travelling alone receive unique groups
What supports or rejects the hypothesis

Supports the hypothesis:

Group-aware CV is around 0.01 or more below ordinary CV
Most grouped folds perform worse
Out-of-fold predictions change meaningfully

That would suggest family leakage was inflating our previous scores.

Rejects the hypothesis:

Group-aware CV remains very close to ordinary CV, roughly within 0.005
Fold stability remains similar
Few out-of-fold predictions change

That would suggest family overlap is not the main reason for the CV–Kaggle gap.

A result between those ranges would be inconclusive rather than a clean win or loss.

In [23]:
# ============================================================
# EXP-006: GROUP-AWARE VALIDATION
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score
from sklearn.model_selection import (
    StratifiedGroupKFold,
    cross_val_predict,
    cross_val_score,
)


# ------------------------------------------------------------
# 1. CREATE FAMILY GROUPS
# ------------------------------------------------------------

validation_data = train_features.copy()

validation_data["Surname"] = (
    validation_data["Name"]
    .str.extract(r"^([^,]+),", expand=False)
    .str.strip()
    .str.lower()
)

# Likely family members share surname and family size.
#
# Passengers travelling alone receive unique groups so that
# unrelated solo passengers with the same surname are not grouped.

validation_data["FamilyGroup"] = np.where(
    validation_data["FamilySize"] > 1,
    (
        validation_data["Surname"]
        + "_family_"
        + validation_data["FamilySize"].astype(int).astype(str)
    ),
    (
        "solo_passenger_"
        + validation_data["PassengerId"].astype(str)
    ),
)

family_groups = validation_data["FamilyGroup"]

print("Number of passengers:", len(validation_data))
print("Number of family groups:", family_groups.nunique())

print(
    "Passengers in multi-person family groups:",
    (validation_data["FamilySize"] > 1).sum(),
)

print(
    "Passengers travelling alone:",
    (validation_data["FamilySize"] == 1).sum(),
)


# ------------------------------------------------------------
# 2. DEFINE GROUP-AWARE VALIDATION
# ------------------------------------------------------------

group_cross_validation = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)


# ------------------------------------------------------------
# 3. CONFIRM THAT GROUPS DO NOT CROSS FOLDS
# ------------------------------------------------------------

fold_summaries = []

for fold_number, (train_indices, valid_indices) in enumerate(
    group_cross_validation.split(
        X,
        y,
        groups=family_groups,
    ),
    start=1,
):
    train_group_values = set(
        family_groups.iloc[train_indices]
    )

    valid_group_values = set(
        family_groups.iloc[valid_indices]
    )

    overlapping_groups = (
        train_group_values
        & valid_group_values
    )

    fold_summaries.append(
        {
            "fold": fold_number,
            "training_rows": len(train_indices),
            "validation_rows": len(valid_indices),
            "validation_survival_rate": (
                y.iloc[valid_indices].mean()
            ),
            "validation_groups": len(
                valid_group_values
            ),
            "group_overlap": len(
                overlapping_groups
            ),
        }
    )

fold_summary = pd.DataFrame(fold_summaries)

print("\nGrouped-fold summary:")
display(fold_summary.round(4))

assert fold_summary["group_overlap"].eq(0).all()


# ------------------------------------------------------------
# 4. ORDINARY STRATIFIED VALIDATION
# ------------------------------------------------------------

ordinary_cv_scores = cross_val_score(
    random_forest_pipeline,
    X,
    y,
    cv=cross_validation,
    scoring="accuracy",
)

print("\nOrdinary StratifiedKFold scores:")
print(ordinary_cv_scores.round(4))

print(
    f"Ordinary mean accuracy: "
    f"{ordinary_cv_scores.mean():.4f}"
)

print(
    f"Ordinary standard deviation: "
    f"{ordinary_cv_scores.std():.4f}"
)


# ------------------------------------------------------------
# 5. GROUP-AWARE VALIDATION
# ------------------------------------------------------------

grouped_cv_scores = cross_val_score(
    random_forest_pipeline,
    X,
    y,
    cv=group_cross_validation,
    groups=family_groups,
    scoring="accuracy",
)

print("\nStratifiedGroupKFold scores:")
print(grouped_cv_scores.round(4))

print(
    f"Grouped mean accuracy: "
    f"{grouped_cv_scores.mean():.4f}"
)

print(
    f"Grouped standard deviation: "
    f"{grouped_cv_scores.std():.4f}"
)


# ------------------------------------------------------------
# 6. OUT-OF-FOLD PREDICTIONS
# ------------------------------------------------------------

ordinary_oof_predictions = cross_val_predict(
    random_forest_pipeline,
    X,
    y,
    cv=cross_validation,
    method="predict",
)

grouped_oof_predictions = cross_val_predict(
    random_forest_pipeline,
    X,
    y,
    cv=group_cross_validation,
    groups=family_groups,
    method="predict",
)

ordinary_oof_accuracy = accuracy_score(
    y,
    ordinary_oof_predictions,
)

grouped_oof_accuracy = accuracy_score(
    y,
    grouped_oof_predictions,
)

oof_disagreements = (
    ordinary_oof_predictions
    != grouped_oof_predictions
).sum()

print(
    f"\nOrdinary OOF accuracy: "
    f"{ordinary_oof_accuracy:.4f}"
)

print(
    f"Grouped OOF accuracy: "
    f"{grouped_oof_accuracy:.4f}"
)

print(
    "Differing out-of-fold predictions:",
    oof_disagreements,
)


# ------------------------------------------------------------
# 7. SUMMARIZE THE VALIDATION COMPARISON
# ------------------------------------------------------------

validation_comparison = pd.DataFrame(
    [
        {
            "validation_method": (
                "5-fold StratifiedKFold"
            ),
            "fold_scores": ", ".join(
                f"{score:.4f}"
                for score in ordinary_cv_scores
            ),
            "mean_accuracy": (
                ordinary_cv_scores.mean()
            ),
            "cv_std": ordinary_cv_scores.std(),
            "oof_accuracy": ordinary_oof_accuracy,
        },
        {
            "validation_method": (
                "5-fold StratifiedGroupKFold"
            ),
            "fold_scores": ", ".join(
                f"{score:.4f}"
                for score in grouped_cv_scores
            ),
            "mean_accuracy": (
                grouped_cv_scores.mean()
            ),
            "cv_std": grouped_cv_scores.std(),
            "oof_accuracy": grouped_oof_accuracy,
        },
    ]
)

ordinary_mean = ordinary_cv_scores.mean()
grouped_mean = grouped_cv_scores.mean()

validation_difference = (
    grouped_mean - ordinary_mean
)

print(
    "\nGrouped minus ordinary mean accuracy:",
    f"{validation_difference:+.4f}",
)

display(
    validation_comparison.style.format(
        {
            "mean_accuracy": "{:.4f}",
            "cv_std": "{:.4f}",
            "oof_accuracy": "{:.4f}",
        }
    )
)


# ------------------------------------------------------------
# 8. SAVE THE VALIDATION RESULTS
# ------------------------------------------------------------

validation_results_path = (
    output_path / "validation_exp006.csv"
)

validation_comparison.to_csv(
    validation_results_path,
    index=False,
)

print("\nSaved validation comparison to:")
print(validation_results_path.resolve())

Number of passengers: 891
Number of family groups: 739
Passengers in multi-person family groups: 354
Passengers travelling alone: 537

Grouped-fold summary:


,fold,training_rows,validation_rows,validation_survival_rate,validation_groups,group_overlap
0,1,716,175,0.4114,144,0
1,2,714,177,0.3390,146,0
2,3,717,174,0.3908,149,0
3,4,703,188,0.4096,150,0
4,5,714,177,0.3672,150,0



Ordinary StratifiedKFold scores:
[0.838  0.8202 0.8202 0.8315 0.8483]
Ordinary mean accuracy: 0.8316
Ordinary standard deviation: 0.0108

StratifiedGroupKFold scores:
[0.7486 0.8588 0.8276 0.8245 0.8757]
Grouped mean accuracy: 0.8270
Grouped standard deviation: 0.0437

Ordinary OOF accuracy: 0.8316
Grouped OOF accuracy: 0.8272
Differing out-of-fold predictions: 12

Grouped minus ordinary mean accuracy: -0.0046


,validation_method,fold_scores,mean_accuracy,cv_std,oof_accuracy
0,5-fold StratifiedKFold,"0.8380, 0.8202, 0.8202, 0.8315, 0.8483",0.8316,0.0108,0.8316
1,5-fold StratifiedGroupKFold,"0.7486, 0.8588, 0.8276, 0.8245, 0.8757",0.8270,0.0437,0.8272



Saved validation comparison to:
C:\Users\Owner\Documents\Github\machine-learning-lab\00-Kaggle\01-Titanic - Machine Learning from Disaster\validation_exp006.csv


##### Experiment Log

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# EXPERIMENT LOGGING FUNCTION
# ============================================================

experiment_path = Path("experiments.csv")


def log_experiment(
    experiment_id,
    description,
    model,
    features,
    validation_method,
    cv_scores,
    kaggle_score,
    changes,
    notes="",
    submission_file="",
):
    """
    Add or update one experiment in experiments.csv.

    Re-running an experiment with the same experiment_id replaces
    the previous record instead of creating a duplicate.
    """

    cv_scores = np.array(cv_scores, dtype=float)

    new_experiment = pd.DataFrame(
        [
            {
                "experiment_id": experiment_id,
                "description": description,
                "model": model,
                "features": features,
                "validation_method": validation_method,
                "cv_scores": ", ".join(
                    f"{score:.4f}" for score in cv_scores
                ),
                "cv_accuracy": cv_scores.mean(),
                "cv_std": cv_scores.std(),
                "kaggle_score": kaggle_score,
                "cv_kaggle_gap": cv_scores.mean() - kaggle_score,
                "changes": changes,
                "submission_file": submission_file,
                "notes": notes,
            }
        ]
    )

    if experiment_path.exists():
        experiments = pd.read_csv(experiment_path)

        # Remove an existing version of this experiment.
        experiments = experiments[
            experiments["experiment_id"] != experiment_id
        ]

        experiments = pd.concat(
            [experiments, new_experiment],
            ignore_index=True,
        )
    else:
        experiments = new_experiment

    experiments = experiments.sort_values(
        "experiment_id"
    ).reset_index(drop=True)

    experiments.to_csv(experiment_path, index=False)

    display(
        experiments.style.format(
            {
                "cv_accuracy": "{:.4f}",
                "cv_std": "{:.4f}",
                "kaggle_score": "{:.5f}",
                "cv_kaggle_gap": "{:.4f}",
            }
        )
    )

    return experiments

In [24]:
FEATURE_LIST = (
    "Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, "
    "Title, FamilySize, IsAlone, Deck, FarePerPerson"
)


# ============================================================
# EXP-001: XGBOOST
# ============================================================

experiments = log_experiment(
    experiment_id="EXP-001",
    description="XGBoost baseline with engineered features",
    model="XGBClassifier",
    features=FEATURE_LIST,
    validation_method="5-fold StratifiedKFold",
    cv_scores=[
        0.8659,
        0.8483,
        0.8034,
        0.8258,
        0.8427,
    ],
    kaggle_score=0.76076,
    changes="Initial engineered-feature baseline",
    submission_file="submission_xgb_baseline.csv",
    notes=(
        "Highest local CV so far, but large CV-to-Kaggle gap. "
        "May be fitting training-specific nonlinear patterns."
    ),
)


# ============================================================
# EXP-002: LOGISTIC REGRESSION
# ============================================================

experiments = log_experiment(
    experiment_id="EXP-002",
    description="Logistic regression using identical engineered features",
    model="LogisticRegression",
    features=FEATURE_LIST,
    validation_method="5-fold StratifiedKFold",
    cv_scores=[
        0.8436,
        0.8258,
        0.7978,
        0.8315,
        0.8427,
    ],
    kaggle_score=0.76555,
    changes=(
        "Replaced XGBClassifier with LogisticRegression; "
        "all other modeling steps held constant"
    ),
    submission_file="submission_logistic.csv",
    notes=(
        "Lower CV than XGBoost but better Kaggle score and "
        "smaller validation-to-leaderboard gap. Current leader."
    ),
)


# ============================================================
# EXP-003: RANDOM FOREST
# ============================================================

experiments = log_experiment(
    experiment_id="EXP-003",
    description="Random forest using identical engineered features",
    model="RandomForestClassifier",
    features=FEATURE_LIST,
    validation_method="5-fold StratifiedKFold",
    cv_scores=[
        0.8380,
        0.8202,
        0.8202,
        0.8315,
        0.8483,
    ],
    kaggle_score=0.78708,
    changes=(
        "Replaced logistic regression with a regularized random forest; "
        "features, preprocessing, and validation folds held constant"
    ),
    submission_file="submission_random_forest.csv",
    notes=(
        "Best Kaggle score so far. Random forest differed from logistic "
        "regression on only 15 of 418 passengers. In 12 of the 15 cases, "
        "logistic predicted survival while random forest predicted death; "
        "10 of those passengers were male and 7 were first-class males. "
        "This suggests random forest captured interactions between sex, "
        "class, title, fare, and deck that additive logistic regression "
        "could not represent. Models agreed on 96.4% of test predictions."
    ),
)


# ============================================================
# EXP-004: LOGISTIC INTERACTIONS
# ============================================================

experiments = log_experiment(
    experiment_id="EXP-004",
    description=(
        "Logistic regression with targeted categorical "
        "interaction features"
    ),
    model="LogisticRegression",
    features=(
        FEATURE_LIST
        + ", Sex_Pclass, Sex_Title, Title_Pclass"
    ),
    validation_method="5-fold StratifiedKFold",
    cv_scores=[
        0.8268,
        0.8539,
        0.7978,
        0.8427,
        0.8483,
    ],
    kaggle_score=0.76315,
    changes=(
        "Added Sex_Pclass, Sex_Title, and Title_Pclass interaction "
        "features to baseline logistic regression; model settings, "
        "preprocessing approach, and validation folds held constant"
    ),
    submission_file="submission_logistic_interactions.csv",
    notes=(
        "Interaction features raised mean CV accuracy from 0.8283 to "
        "0.8339 but reduced Kaggle accuracy from 0.76555 to 0.76315. "
        "The CV standard deviation and CV-to-Kaggle gap also increased. "
        "The targeted interactions therefore fit the training folds "
        "better without transferring to the hidden test passengers. "
        "Random Forest remains the best model at 0.78708."
    ),
)


# ============================================================
# EXP-005: RF age imputation
# ============================================================

experiments = log_experiment(
    experiment_id="EXP-005",
    description=(
        "Random forest with title-and-class-based age imputation"
    ),
    model="RandomForestClassifier",
    features=FEATURE_LIST,
    validation_method="5-fold StratifiedKFold",
    cv_scores=[
        0.8380,
        0.8146,
        0.8258,
        0.8315,
        0.8371,
    ],
    kaggle_score=0.78229,
    changes=(
        "Replaced overall median age imputation with fold-safe grouped "
        "median imputation using Title and Pclass; random forest settings, "
        "features, and validation folds held constant"
    ),
    submission_file="submission_rf_grouped_age.csv",
    notes=(
        "Grouped age imputation reduced mean CV accuracy from 0.8316 "
        "to 0.8294 and Kaggle accuracy from 0.78708 to 0.78229. "
        "It changed only 4 of 418 test predictions versus EXP-003. "
        "Although CV variability decreased slightly, the changed "
        "predictions did not improve hidden-test performance. "
        "Reject EXP-005; EXP-003 remains champion."
    ),
)


# ============================================================
# EXP-006
# ============================================================

experiments = log_experiment(
    experiment_id="EXP-006",
    description=(
        "Compared ordinary stratified validation with "
        "family-group-aware validation"
    ),
    model="RandomForestClassifier",
    features=FEATURE_LIST,
    validation_method=(
        "5-fold StratifiedGroupKFold using surname and family size"
    ),
    cv_scores=[
        0.7486,
        0.8588,
        0.8276,
        0.8245,
        0.8757,
    ],
    kaggle_score=np.nan,
    changes=(
        "Held the EXP-003 Random Forest, features, and preprocessing "
        "constant; replaced StratifiedKFold with StratifiedGroupKFold. "
        "Likely family members were assigned to the same fold using "
        "surname and family size, while solo passengers received "
        "individual groups."
    ),
    submission_file="",
    notes=(
        "No Kaggle submission was needed because the final trained model "
        "and test predictions did not change. Group-aware mean accuracy "
        "was 0.8270 versus 0.8316 for ordinary validation, a reduction "
        "of only 0.0046. This does not support family leakage as the main "
        "cause of the CV-to-Kaggle gap. However, grouped validation was "
        "much less stable, with SD 0.0437 versus 0.0108. Only 12 of 891 "
        "out-of-fold predictions differed. Retain StratifiedKFold as the "
        "primary comparison method and use grouped validation as a "
        "secondary robustness check."
    ),
)

,experiment_id,description,model,features,validation_method,cv_accuracy,kaggle_score,notes,filename,cv_scores,cv_std,cv_kaggle_gap,changes,submission_file
0,EXP-001,XGBoost baseline with engineered features,XGBClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8372,0.76076,"Highest local CV so far, but large CV-to-Kaggle gap. May be fitting training-specific nonlinear patterns.",nan,"0.8659, 0.8483, 0.8034, 0.8258, 0.8427",0.0212,0.0765,Initial engineered-feature baseline,submission_xgb_baseline.csv
1,EXP-002,Logistic regression using identical engineered features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8283,0.76555,Lower CV than XGBoost but better Kaggle score and smaller validation-to-leaderboard gap. Current leader.,nan,"0.8436, 0.8258, 0.7978, 0.8315, 0.8427",0.0167,0.0627,Replaced XGBClassifier with LogisticRegression; all other modeling steps held constant,submission_logistic.csv
2,EXP-003,Random forest using identical engineered features,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8316,0.78708,"Best Kaggle score so far. Random forest differed from logistic regression on only 15 of 418 passengers. In 12 of the 15 cases, logistic predicted survival while random forest predicted death; 10 of those passengers were male and 7 were first-class males. This suggests random forest captured interactions between sex, class, title, fare, and deck that additive logistic regression could not represent. Models agreed on 96.4% of test predictions.",nan,"0.8380, 0.8202, 0.8202, 0.8315, 0.8483",0.0108,0.0446,"Replaced logistic regression with a regularized random forest; features, preprocessing, and validation folds held constant",submission_random_forest.csv
3,EXP-004,Logistic regression with targeted categorical interaction features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson, Sex_Pclass, Sex_Title, Title_Pclass",5-fold StratifiedKFold,0.8339,0.76315,Interaction features raised mean CV accuracy from 0.8283 to 0.8339 but reduced Kaggle accuracy from 0.76555 to 0.76315. The CV standard deviation and CV-to-Kaggle gap also increased. The targeted interactions therefore fit the training folds better without transferring to the hidden test passengers. Random Forest remains the best model at 0.78708.,nan,"0.8268, 0.8539, 0.7978, 0.8427, 0.8483",0.0202,0.0707,"Added Sex_Pclass, Sex_Title, and Title_Pclass interaction features to baseline logistic regression; model settings, preprocessing approach, and validation folds held constant",submission_logistic_interactions.csv
4,EXP-005,Random forest with title-and-class-based age imputation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8294,0.78229,"Grouped age imputation reduced mean CV accuracy from 0.8316 to 0.8294 and Kaggle accuracy from 0.78708 to 0.78229. It changed only 4 of 418 test predictions versus EXP-003. Although CV variability decreased slightly, the changed predictions did not improve hidden-test performance. Reject EXP-005; EXP-003 remains champion.",nan,"0.8380, 0.8146, 0.8258, 0.8315, 0.8371",0.0086,0.0471,"Replaced overall median age imputation with fold-safe grouped median imputation using Title and Pclass; random forest settings, features, and validation folds held constant",submission_rf_grouped_age.csv


,experiment_id,description,model,features,validation_method,cv_accuracy,kaggle_score,notes,filename,cv_scores,cv_std,cv_kaggle_gap,changes,submission_file
0,EXP-001,XGBoost baseline with engineered features,XGBClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8372,0.76076,"Highest local CV so far, but large CV-to-Kaggle gap. May be fitting training-specific nonlinear patterns.",nan,"0.8659, 0.8483, 0.8034, 0.8258, 0.8427",0.0212,0.0765,Initial engineered-feature baseline,submission_xgb_baseline.csv
1,EXP-002,Logistic regression using identical engineered features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8283,0.76555,Lower CV than XGBoost but better Kaggle score and smaller validation-to-leaderboard gap. Current leader.,nan,"0.8436, 0.8258, 0.7978, 0.8315, 0.8427",0.0167,0.0627,Replaced XGBClassifier with LogisticRegression; all other modeling steps held constant,submission_logistic.csv
2,EXP-003,Random forest using identical engineered features,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8316,0.78708,"Best Kaggle score so far. Random forest differed from logistic regression on only 15 of 418 passengers. In 12 of the 15 cases, logistic predicted survival while random forest predicted death; 10 of those passengers were male and 7 were first-class males. This suggests random forest captured interactions between sex, class, title, fare, and deck that additive logistic regression could not represent. Models agreed on 96.4% of test predictions.",nan,"0.8380, 0.8202, 0.8202, 0.8315, 0.8483",0.0108,0.0446,"Replaced logistic regression with a regularized random forest; features, preprocessing, and validation folds held constant",submission_random_forest.csv
3,EXP-004,Logistic regression with targeted categorical interaction features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson, Sex_Pclass, Sex_Title, Title_Pclass",5-fold StratifiedKFold,0.8339,0.76315,Interaction features raised mean CV accuracy from 0.8283 to 0.8339 but reduced Kaggle accuracy from 0.76555 to 0.76315. The CV standard deviation and CV-to-Kaggle gap also increased. The targeted interactions therefore fit the training folds better without transferring to the hidden test passengers. Random Forest remains the best model at 0.78708.,nan,"0.8268, 0.8539, 0.7978, 0.8427, 0.8483",0.0202,0.0707,"Added Sex_Pclass, Sex_Title, and Title_Pclass interaction features to baseline logistic regression; model settings, preprocessing approach, and validation folds held constant",submission_logistic_interactions.csv
4,EXP-005,Random forest with title-and-class-based age imputation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8294,0.78229,"Grouped age imputation reduced mean CV accuracy from 0.8316 to 0.8294 and Kaggle accuracy from 0.78708 to 0.78229. It changed only 4 of 418 test predictions versus EXP-003. Although CV variability decreased slightly, the changed predictions did not improve hidden-test performance. Reject EXP-005; EXP-003 remains champion.",nan,"0.8380, 0.8146, 0.8258, 0.8315, 0.8371",0.0086,0.0471,"Replaced overall median age imputation with fold-safe grouped median imputation using Title and Pclass; random forest settings, features, and validation folds held constant",submission_rf_grouped_age.csv


,experiment_id,description,model,features,validation_method,cv_accuracy,kaggle_score,notes,filename,cv_scores,cv_std,cv_kaggle_gap,changes,submission_file
0,EXP-001,XGBoost baseline with engineered features,XGBClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8372,0.76076,"Highest local CV so far, but large CV-to-Kaggle gap. May be fitting training-specific nonlinear patterns.",nan,"0.8659, 0.8483, 0.8034, 0.8258, 0.8427",0.0212,0.0765,Initial engineered-feature baseline,submission_xgb_baseline.csv
1,EXP-002,Logistic regression using identical engineered features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8283,0.76555,Lower CV than XGBoost but better Kaggle score and smaller validation-to-leaderboard gap. Current leader.,nan,"0.8436, 0.8258, 0.7978, 0.8315, 0.8427",0.0167,0.0627,Replaced XGBClassifier with LogisticRegression; all other modeling steps held constant,submission_logistic.csv
2,EXP-003,Random forest using identical engineered features,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8316,0.78708,"Best Kaggle score so far. Random forest differed from logistic regression on only 15 of 418 passengers. In 12 of the 15 cases, logistic predicted survival while random forest predicted death; 10 of those passengers were male and 7 were first-class males. This suggests random forest captured interactions between sex, class, title, fare, and deck that additive logistic regression could not represent. Models agreed on 96.4% of test predictions.",nan,"0.8380, 0.8202, 0.8202, 0.8315, 0.8483",0.0108,0.0446,"Replaced logistic regression with a regularized random forest; features, preprocessing, and validation folds held constant",submission_random_forest.csv
3,EXP-004,Logistic regression with targeted categorical interaction features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson, Sex_Pclass, Sex_Title, Title_Pclass",5-fold StratifiedKFold,0.8339,0.76315,Interaction features raised mean CV accuracy from 0.8283 to 0.8339 but reduced Kaggle accuracy from 0.76555 to 0.76315. The CV standard deviation and CV-to-Kaggle gap also increased. The targeted interactions therefore fit the training folds better without transferring to the hidden test passengers. Random Forest remains the best model at 0.78708.,nan,"0.8268, 0.8539, 0.7978, 0.8427, 0.8483",0.0202,0.0707,"Added Sex_Pclass, Sex_Title, and Title_Pclass interaction features to baseline logistic regression; model settings, preprocessing approach, and validation folds held constant",submission_logistic_interactions.csv
4,EXP-005,Random forest with title-and-class-based age imputation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8294,0.78229,"Grouped age imputation reduced mean CV accuracy from 0.8316 to 0.8294 and Kaggle accuracy from 0.78708 to 0.78229. It changed only 4 of 418 test predictions versus EXP-003. Although CV variability decreased slightly, the changed predictions did not improve hidden-test performance. Reject EXP-005; EXP-003 remains champion.",nan,"0.8380, 0.8146, 0.8258, 0.8315, 0.8371",0.0086,0.0471,"Replaced overall median age imputation with fold-safe grouped median imputation using Title and Pclass; random forest settings, features, and validation folds held constant",submission_rf_grouped_age.csv


,experiment_id,description,model,features,validation_method,cv_accuracy,kaggle_score,notes,filename,cv_scores,cv_std,cv_kaggle_gap,changes,submission_file
0,EXP-001,XGBoost baseline with engineered features,XGBClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8372,0.76076,"Highest local CV so far, but large CV-to-Kaggle gap. May be fitting training-specific nonlinear patterns.",nan,"0.8659, 0.8483, 0.8034, 0.8258, 0.8427",0.0212,0.0765,Initial engineered-feature baseline,submission_xgb_baseline.csv
1,EXP-002,Logistic regression using identical engineered features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8283,0.76555,Lower CV than XGBoost but better Kaggle score and smaller validation-to-leaderboard gap. Current leader.,nan,"0.8436, 0.8258, 0.7978, 0.8315, 0.8427",0.0167,0.0627,Replaced XGBClassifier with LogisticRegression; all other modeling steps held constant,submission_logistic.csv
2,EXP-003,Random forest using identical engineered features,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8316,0.78708,"Best Kaggle score so far. Random forest differed from logistic regression on only 15 of 418 passengers. In 12 of the 15 cases, logistic predicted survival while random forest predicted death; 10 of those passengers were male and 7 were first-class males. This suggests random forest captured interactions between sex, class, title, fare, and deck that additive logistic regression could not represent. Models agreed on 96.4% of test predictions.",nan,"0.8380, 0.8202, 0.8202, 0.8315, 0.8483",0.0108,0.0446,"Replaced logistic regression with a regularized random forest; features, preprocessing, and validation folds held constant",submission_random_forest.csv
3,EXP-004,Logistic regression with targeted categorical interaction features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson, Sex_Pclass, Sex_Title, Title_Pclass",5-fold StratifiedKFold,0.8339,0.76315,Interaction features raised mean CV accuracy from 0.8283 to 0.8339 but reduced Kaggle accuracy from 0.76555 to 0.76315. The CV standard deviation and CV-to-Kaggle gap also increased. The targeted interactions therefore fit the training folds better without transferring to the hidden test passengers. Random Forest remains the best model at 0.78708.,nan,"0.8268, 0.8539, 0.7978, 0.8427, 0.8483",0.0202,0.0708,"Added Sex_Pclass, Sex_Title, and Title_Pclass interaction features to baseline logistic regression; model settings, preprocessing approach, and validation folds held constant",submission_logistic_interactions.csv
4,EXP-005,Random forest with title-and-class-based age imputation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8294,0.78229,"Grouped age imputation reduced mean CV accuracy from 0.8316 to 0.8294 and Kaggle accuracy from 0.78708 to 0.78229. It changed only 4 of 418 test predictions versus EXP-003. Although CV variability decreased slightly, the changed predictions did not improve hidden-test performance. Reject EXP-005; EXP-003 remains champion.",nan,"0.8380, 0.8146, 0.8258, 0.8315, 0.8371",0.0086,0.0471,"Replaced overall median age imputation with fold-safe grouped median imputation using Title and Pclass; random forest settings, features, and validation folds held constant",submission_rf_grouped_age.csv


,experiment_id,description,model,features,validation_method,cv_accuracy,kaggle_score,notes,filename,cv_scores,cv_std,cv_kaggle_gap,changes,submission_file
0,EXP-001,XGBoost baseline with engineered features,XGBClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8372,0.76076,"Highest local CV so far, but large CV-to-Kaggle gap. May be fitting training-specific nonlinear patterns.",nan,"0.8659, 0.8483, 0.8034, 0.8258, 0.8427",0.0212,0.0765,Initial engineered-feature baseline,submission_xgb_baseline.csv
1,EXP-002,Logistic regression using identical engineered features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8283,0.76555,Lower CV than XGBoost but better Kaggle score and smaller validation-to-leaderboard gap. Current leader.,nan,"0.8436, 0.8258, 0.7978, 0.8315, 0.8427",0.0167,0.0627,Replaced XGBClassifier with LogisticRegression; all other modeling steps held constant,submission_logistic.csv
2,EXP-003,Random forest using identical engineered features,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8316,0.78708,"Best Kaggle score so far. Random forest differed from logistic regression on only 15 of 418 passengers. In 12 of the 15 cases, logistic predicted survival while random forest predicted death; 10 of those passengers were male and 7 were first-class males. This suggests random forest captured interactions between sex, class, title, fare, and deck that additive logistic regression could not represent. Models agreed on 96.4% of test predictions.",nan,"0.8380, 0.8202, 0.8202, 0.8315, 0.8483",0.0108,0.0446,"Replaced logistic regression with a regularized random forest; features, preprocessing, and validation folds held constant",submission_random_forest.csv
3,EXP-004,Logistic regression with targeted categorical interaction features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson, Sex_Pclass, Sex_Title, Title_Pclass",5-fold StratifiedKFold,0.8339,0.76315,Interaction features raised mean CV accuracy from 0.8283 to 0.8339 but reduced Kaggle accuracy from 0.76555 to 0.76315. The CV standard deviation and CV-to-Kaggle gap also increased. The targeted interactions therefore fit the training folds better without transferring to the hidden test passengers. Random Forest remains the best model at 0.78708.,nan,"0.8268, 0.8539, 0.7978, 0.8427, 0.8483",0.0202,0.0707,"Added Sex_Pclass, Sex_Title, and Title_Pclass interaction features to baseline logistic regression; model settings, preprocessing approach, and validation folds held constant",submission_logistic_interactions.csv
4,EXP-005,Random forest with title-and-class-based age imputation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8294,0.78229,"Grouped age imputation reduced mean CV accuracy from 0.8316 to 0.8294 and Kaggle accuracy from 0.78708 to 0.78229. It changed only 4 of 418 test predictions versus EXP-003. Although CV variability decreased slightly, the changed predictions did not improve hidden-test performance. Reject EXP-005; EXP-003 remains champion.",nan,"0.8380, 0.8146, 0.8258, 0.8315, 0.8371",0.0086,0.0471,"Replaced overall median age imputation with fold-safe grouped median imputation using Title and Pclass; random forest settings, features, and validation folds held constant",submission_rf_grouped_age.csv


,experiment_id,description,model,features,validation_method,cv_accuracy,kaggle_score,notes,filename,cv_scores,cv_std,cv_kaggle_gap,changes,submission_file
0,EXP-001,XGBoost baseline with engineered features,XGBClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8372,0.76076,"Highest local CV so far, but large CV-to-Kaggle gap. May be fitting training-specific nonlinear patterns.",nan,"0.8659, 0.8483, 0.8034, 0.8258, 0.8427",0.0212,0.0765,Initial engineered-feature baseline,submission_xgb_baseline.csv
1,EXP-002,Logistic regression using identical engineered features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8283,0.76555,Lower CV than XGBoost but better Kaggle score and smaller validation-to-leaderboard gap. Current leader.,nan,"0.8436, 0.8258, 0.7978, 0.8315, 0.8427",0.0167,0.0627,Replaced XGBClassifier with LogisticRegression; all other modeling steps held constant,submission_logistic.csv
2,EXP-003,Random forest using identical engineered features,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8316,0.78708,"Best Kaggle score so far. Random forest differed from logistic regression on only 15 of 418 passengers. In 12 of the 15 cases, logistic predicted survival while random forest predicted death; 10 of those passengers were male and 7 were first-class males. This suggests random forest captured interactions between sex, class, title, fare, and deck that additive logistic regression could not represent. Models agreed on 96.4% of test predictions.",nan,"0.8380, 0.8202, 0.8202, 0.8315, 0.8483",0.0108,0.0446,"Replaced logistic regression with a regularized random forest; features, preprocessing, and validation folds held constant",submission_random_forest.csv
3,EXP-004,Logistic regression with targeted categorical interaction features,LogisticRegression,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson, Sex_Pclass, Sex_Title, Title_Pclass",5-fold StratifiedKFold,0.8339,0.76315,Interaction features raised mean CV accuracy from 0.8283 to 0.8339 but reduced Kaggle accuracy from 0.76555 to 0.76315. The CV standard deviation and CV-to-Kaggle gap also increased. The targeted interactions therefore fit the training folds better without transferring to the hidden test passengers. Random Forest remains the best model at 0.78708.,nan,"0.8268, 0.8539, 0.7978, 0.8427, 0.8483",0.0202,0.0707,"Added Sex_Pclass, Sex_Title, and Title_Pclass interaction features to baseline logistic regression; model settings, preprocessing approach, and validation folds held constant",submission_logistic_interactions.csv
4,EXP-005,Random forest with title-and-class-based age imputation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedKFold,0.8294,0.78229,"Grouped age imputation reduced mean CV accuracy from 0.8316 to 0.8294 and Kaggle accuracy from 0.78708 to 0.78229. It changed only 4 of 418 test predictions versus EXP-003. Although CV variability decreased slightly, the changed predictions did not improve hidden-test performance. Reject EXP-005; EXP-003 remains champion.",nan,"0.8380, 0.8146, 0.8258, 0.8315, 0.8371",0.0086,0.0471,"Replaced overall median age imputation with fold-safe grouped median imputation using Title and Pclass; random forest settings, features, and validation folds held constant",submission_rf_grouped_age.csv
5,EXP-006,Compared ordinary stratified validation with family-group-aware validation,RandomForestClassifier,"Pclass, Sex, Age, SibSp, Parch, Fare, Embarked, Title, FamilySize, IsAlone, Deck, FarePerPerson",5-fold StratifiedGroupKFold using surname and family size,0.8270,nan,"No Kaggle submission was needed because the final tra